In [18]:
import json
from urllib.request import Request, urlopen
TEAM_ID = "TEAM_33"
API_KEY = "oc_WZ1MdJATkHiGb6KPTlyXfL4p4l9x37Qk"
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle"
def api(payload):
    req = Request(API_URL, data=json.dumps(payload).encode(),

headers={"Content-Type":"application/json", "X-API-Key":API_KEY})

    with urlopen(req, timeout=30) as r:
        return json.load(r)
spec = api({"action":"spec", "team_id":TEAM_ID})
SPACE = spec["hyperparameters"]
print("Model:", spec["model"])
for name, values in SPACE.items():
    print(name, ":", values)

Model: SGDRegressor
eta0 : [0.0001, 0.001, 0.01, 0.05]
loss : ['squared_error', 'huber', 'epsilon_insensitive']
alpha : [1e-06, 3.727593720314938e-06, 1.389495494373136e-05, 5.1794746792312125e-05, 0.00019306977288832496, 0.0007196856730011514, 0.0026826957952797246, 0.01]
average : [False, True]
penalty : ['l2', 'l1', 'elasticnet']
learning_rate : ['constant', 'invscaling', 'adaptive', 'optimal']


In [19]:
# Assigned search space

search_space = {
    "eta0": [
        0.0001,
        0.001,
        0.01,
        0.05
    ],

    "loss": [
        "squared_error",
        "huber",
        "epsilon_insensitive"
    ],

    "alpha": [
        1e-06,
        3.727593720314938e-06,
        1.389495494373136e-05,
        5.1794746792312125e-05,
        0.00019306977288832496,
        0.0007196856730011514,
        0.0026826957952797246,
        0.01
    ],

    "average": [
        False,
        True
    ],

    "penalty": [
        "l2",
        "l1",
        "elasticnet"
    ],

    "learning_rate": [
        "constant",
        "invscaling",
        "adaptive",
        "optimal"
    ]
}


# Count all possible combinations

total_configurations = 1

for values in search_space.values():
    total_configurations *= len(values)

print("\nTotal possible configurations:", total_configurations)


# Required Oracle function

def oracle_query(params):

    result = api({
        "action": "query",
        "team_id": TEAM_ID,
        "params": params
    })

    return float(result["loss"])


Total possible configurations: 2304


In [20]:
# Store every Oracle experiment

history = []

# Number of Oracle calls made
oracle_calls = 0

# Configurations already tested
evaluated = set()


def configuration_key(params):
    return (
        params["eta0"],
        params["loss"],
        params["alpha"],
        params["average"],
        params["penalty"],
        params["learning_rate"]
    )


def evaluate(params):

    global oracle_calls

    key = configuration_key(params)

    # Do not waste an Oracle call on a duplicate configuration
    if key in evaluated:
        return None

    evaluated.add(key)

    loss = oracle_query(params)

    oracle_calls += 1

    result = {
        "call": oracle_calls,
        "params": params.copy(),
        "loss": loss
    }

    history.append(result)

    print(
        f"Call {oracle_calls:3d} | "
        f"Loss = {loss:.6f} | "
        f"{params}"
    )

    return loss


def random_configuration():

    params = {}

    for parameter, values in search_space.items():
        params[parameter] = random.choice(values)

    return params


def get_best():

    if not history:
        return None

    return min(
        history,
        key=lambda result: result["loss"]
    )

In [21]:
# ---------------------------------------------------------
# PHASE 1: RANDOM EXPLORATION
# ---------------------------------------------------------

INITIAL_TRIALS = 20

print("=" * 80)
print("PHASE 1: RANDOM EXPLORATION")
print("=" * 80)

while oracle_calls < INITIAL_TRIALS:

    params = random_configuration()

    evaluate(params)


best = get_best()

print("\nCurrent best after random exploration:")
print("Loss:", best["loss"])
print("Parameters:", best["params"])

PHASE 1: RANDOM EXPLORATION
Call   1 | Loss = 71.821450 | {'eta0': 0.0001, 'loss': 'squared_error', 'alpha': 0.00019306977288832496, 'average': False, 'penalty': 'l2', 'learning_rate': 'invscaling'}
Call   2 | Loss = 131.788389 | {'eta0': 0.0001, 'loss': 'epsilon_insensitive', 'alpha': 3.727593720314938e-06, 'average': True, 'penalty': 'l2', 'learning_rate': 'constant'}
Call   3 | Loss = 71.820754 | {'eta0': 0.0001, 'loss': 'squared_error', 'alpha': 5.1794746792312125e-05, 'average': False, 'penalty': 'elasticnet', 'learning_rate': 'invscaling'}
Call   4 | Loss = 14.510487 | {'eta0': 0.05, 'loss': 'squared_error', 'alpha': 0.01, 'average': True, 'penalty': 'l2', 'learning_rate': 'invscaling'}
Call   5 | Loss = 14.555291 | {'eta0': 0.05, 'loss': 'huber', 'alpha': 0.00019306977288832496, 'average': False, 'penalty': 'l2', 'learning_rate': 'adaptive'}
Call   6 | Loss = 14.532604 | {'eta0': 0.0001, 'loss': 'squared_error', 'alpha': 0.0026826957952797246, 'average': False, 'penalty': 'l1', 

In [22]:
# ---------------------------------------------------------
# PHASE 2: ADAPTIVE LOCAL SEARCH
# ---------------------------------------------------------

MAX_CALLS = 80


def generate_neighbors(params):

    neighbors = []

    # Change ONE hyperparameter at a time
    for parameter in search_space:

        for value in search_space[parameter]:

            if value != params[parameter]:

                new_params = params.copy()
                new_params[parameter] = value

                neighbors.append(new_params)

    return neighbors


print("\n" + "=" * 80)
print("PHASE 2: ADAPTIVE SEARCH")
print("=" * 80)


while oracle_calls < MAX_CALLS:

    # Find the best configuration discovered so far
    best = get_best()

    # Generate configurations close to the current best
    neighbors = generate_neighbors(best["params"])

    random.shuffle(neighbors)

    next_params = None

    # Prefer an unexplored neighboring configuration
    for candidate in neighbors:

        if configuration_key(candidate) not in evaluated:

            next_params = candidate
            break

    # If all nearby configurations were already tested,
    # perform random exploration.
    if next_params is None:

        next_params = random_configuration()

        while configuration_key(next_params) in evaluated:
            next_params = random_configuration()

    evaluate(next_params)


best = get_best()

print("\nCurrent best:")
print("Loss:", best["loss"])
print("Parameters:", best["params"])


PHASE 2: ADAPTIVE SEARCH
Call  21 | Loss = 3182.597761 | {'eta0': 0.01, 'loss': 'epsilon_insensitive', 'alpha': 1e-06, 'average': False, 'penalty': 'elasticnet', 'learning_rate': 'optimal'}
Call  22 | Loss = 14.324872 | {'eta0': 0.05, 'loss': 'epsilon_insensitive', 'alpha': 1e-06, 'average': False, 'penalty': 'elasticnet', 'learning_rate': 'adaptive'}
Call  23 | Loss = 51.213439 | {'eta0': 0.0001, 'loss': 'epsilon_insensitive', 'alpha': 1e-06, 'average': False, 'penalty': 'elasticnet', 'learning_rate': 'adaptive'}
Call  24 | Loss = 14.326829 | {'eta0': 0.01, 'loss': 'epsilon_insensitive', 'alpha': 0.0026826957952797246, 'average': False, 'penalty': 'elasticnet', 'learning_rate': 'adaptive'}
Call  25 | Loss = 14.801243 | {'eta0': 0.01, 'loss': 'epsilon_insensitive', 'alpha': 1e-06, 'average': False, 'penalty': 'elasticnet', 'learning_rate': 'invscaling'}
Call  26 | Loss = 14.350574 | {'eta0': 0.01, 'loss': 'epsilon_insensitive', 'alpha': 1e-06, 'average': False, 'penalty': 'elasticnet'

In [23]:
# ---------------------------------------------------------
# FINAL RESULTS
# ---------------------------------------------------------

best = get_best()

print("\n" + "=" * 80)
print("FINAL RESULT")
print("=" * 80)

print("\nModel:")
print(spec["model"])

print("\nBest Hyperparameters:")

for parameter, value in best["params"].items():
    print(f"  {parameter}: {value}")

print("\nBest Loss:")
print(best["loss"])

print("\nTotal Oracle Calls:")
print(oracle_calls)

print("\nTotal Possible Configurations:")
print(total_configurations)

print("\nOracle Calls Saved:")
print(total_configurations - oracle_calls)


# ---------------------------------------------------------
# TOP 10 CONFIGURATIONS
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("TOP 10 CONFIGURATIONS")
print("=" * 80)

sorted_results = sorted(
    history,
    key=lambda result: result["loss"]
)

for rank, result in enumerate(sorted_results[:10], start=1):

    print(f"\nRank {rank}")
    print("Loss:", result["loss"])
    print("Parameters:", result["params"])


# ---------------------------------------------------------
# COMPLETE EXPERIMENT HISTORY
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("ALL ORACLE CALLS")
print("=" * 80)

for result in history:

    print(
        f"Call {result['call']:3d} | "
        f"Loss = {result['loss']:.6f} | "
        f"{result['params']}"
    )


FINAL RESULT

Model:
SGDRegressor

Best Hyperparameters:
  eta0: 0.01
  loss: squared_error
  alpha: 1e-06
  average: False
  penalty: l2
  learning_rate: adaptive

Best Loss:
14.3036708137427

Total Oracle Calls:
80

Total Possible Configurations:
2304

Oracle Calls Saved:
2224

TOP 10 CONFIGURATIONS

Rank 1
Loss: 14.3036708137427
Parameters: {'eta0': 0.01, 'loss': 'squared_error', 'alpha': 1e-06, 'average': False, 'penalty': 'l2', 'learning_rate': 'adaptive'}

Rank 2
Loss: 14.3036772292947
Parameters: {'eta0': 0.01, 'loss': 'squared_error', 'alpha': 1e-06, 'average': False, 'penalty': 'elasticnet', 'learning_rate': 'adaptive'}

Rank 3
Loss: 14.30368471514
Parameters: {'eta0': 0.01, 'loss': 'squared_error', 'alpha': 3.727593720314938e-06, 'average': False, 'penalty': 'l2', 'learning_rate': 'adaptive'}

Rank 4
Loss: 14.3037136351502
Parameters: {'eta0': 0.01, 'loss': 'squared_error', 'alpha': 1e-06, 'average': False, 'penalty': 'l1', 'learning_rate': 'adaptive'}

Rank 5
Loss: 14.30373

# Black-Box Hyperparameter Optimization Approach

## 1. Overview

The objective of this task is to find a hyperparameter configuration that produces the **lowest possible loss** while using as few Oracle calls as possible.

The model assigned for this task is **SGDRegressor**, with a search space containing six hyperparameters:

- `eta0`
- `loss`
- `alpha`
- `average`
- `penalty`
- `learning_rate`

There are a total of **2304 possible hyperparameter combinations**. Evaluating every combination would require 2304 Oracle calls, so an optimization strategy was developed to explore the search space more efficiently.

The Oracle is treated as a **black box**. For every valid hyperparameter configuration supplied to it, the Oracle returns a loss value. Since lower loss is better, the optimization process attempts to minimize this value.

---

## 2. Optimization Strategy

A two-phase optimization strategy was used:

### Phase 1 — Random Exploration

The first phase performs **random exploration** of the search space.

A number of configurations are selected randomly, and each configuration is submitted to the Oracle. The resulting loss is recorded along with the corresponding hyperparameters.

The purpose of this phase is to obtain an initial understanding of the search space without evaluating all 2304 possible configurations.

Random exploration also reduces the chance of starting the optimization from a poor region of the search space.

---

### Phase 2 — Adaptive Local Search

After the initial random exploration, the best configuration found so far is selected.

New configurations are then generated by changing **one hyperparameter at a time** while keeping the remaining hyperparameters unchanged.

For example, if the current best configuration is:

```text
eta0 = 0.01
loss = squared_error
alpha = 0.000193
average = True
penalty = l2
learning_rate = adaptive
```

the algorithm may generate a neighboring configuration by changing only `eta0`:

```text
eta0 = 0.05
loss = squared_error
alpha = 0.000193
average = True
penalty = l2
learning_rate = adaptive
```

This allows the search to investigate configurations close to a promising solution.

After every Oracle call, the best configuration discovered so far is updated.

---

## 3. Duplicate Configuration Handling

Every Oracle call counts toward the total number of calls, including repeated valid configurations.

Therefore, previously evaluated configurations are stored using a unique configuration key.

Before making an Oracle call, the algorithm checks whether that configuration has already been evaluated.

If it has already been evaluated, another Oracle call is avoided.

This prevents unnecessary consumption of the limited Oracle budget.

---

## 4. Exploration vs. Exploitation

The optimization approach balances two ideas:

### Exploration

Random configurations are used to explore different parts of the search space.

This helps prevent the search from becoming restricted to a small region too early.

### Exploitation

Once a promising configuration is found, its neighboring configurations are investigated.

This allows the algorithm to focus Oracle calls around configurations that have already demonstrated relatively low loss.

Therefore, the overall approach can be summarized as:

```text
Random Exploration
        ↓
Find Current Best
        ↓
Generate Nearby Configurations
        ↓
Evaluate Promising Neighbors
        ↓
Update Best Configuration
        ↓
Repeat
```

---

## 5. Oracle Call Budget

The complete search space contains:

**4 × 3 × 8 × 2 × 3 × 4 = 2304 configurations**

Instead of evaluating all 2304 configurations, the implementation uses a limited Oracle-call budget.

The current implementation performs:

- **20 initial random trials**
- followed by adaptive local search
- with a maximum of **80 total Oracle calls**

Thus, the optimization attempts to find a low-loss configuration using substantially fewer Oracle calls than exhaustive search.

---

## 6. Selection of the Final Configuration

After the optimization process terminates, all evaluated configurations are compared using their Oracle loss.

The configuration with the minimum observed loss is selected as the final result.

The notebook reports:

- Best hyperparameter configuration
- Best observed loss
- Total Oracle calls used
- Total possible configurations
- Oracle calls saved
- Top-performing configurations discovered during the search

---

## 7. Why This Approach Was Used

The assignment requires writing our **own logic for selecting the next hyperparameter configuration** rather than using an automated hyperparameter-optimization library.

Therefore, the implementation does not use tools such as:

- Optuna
- Hyperopt
- GridSearchCV
- RandomizedSearchCV
- Bayesian optimization libraries

Instead, the optimization logic is implemented directly using Python.

The approach combines **random exploration** with **local search around promising configurations**, allowing the search to explore the large configuration space while keeping the number of Oracle calls relatively small.

---

## 8. Summary

The complete optimization procedure is:

1. Obtain the assigned model and hyperparameter search space from the Oracle.
2. Determine the total number of possible configurations.
3. Randomly evaluate an initial set of configurations.
4. Identify the configuration with the lowest observed loss.
5. Generate neighboring configurations by changing one hyperparameter at a time.
6. Evaluate previously unexplored neighbors.
7. Update the best configuration whenever a lower loss is found.
8. Avoid duplicate Oracle calls.
9. Stop when the Oracle-call budget is reached.
10. Report the configuration with the lowest observed loss.

This provides a simple, custom **black-box hyperparameter optimization** strategy that attempts to minimize loss while reducing the number of Oracle evaluations required.